In [8]:
from pathlib import Path
import duckdb
import pandas as pd

DATA_DIR = Path.home() / "Datasets"

con = duckdb.connect()

detectors = con.sql("""
    SELECT *
    FROM read_csv_auto(?)
    LIMIT 10
""", params=[str(DATA_DIR / "detectors.csv")]).df()

display(detectors)

,detid,length,pos,fclass,road,limit,citycode,lanes,linkid,long,lat
0,U1-52G,0.196037,0.005512,secondary,Gögginger Straße,50,augsburg,1,72,10.889553,48.359957
1,U1-51G,0.130039,0.004013,secondary,Gögginger Straße,50,augsburg,1,73,10.889601,48.359945
2,U1-52L,0.155863,0.022228,secondary,Gögginger Straße,50,augsburg,1,70,10.889356,48.359876
3,U1-51L,0.197675,0.021889,secondary,Gögginger Straße,50,augsburg,1,71,10.889396,48.359862
4,U1-62,0.065183,0.024465,secondary,Rosenaustraße,50,augsburg,1,68,10.889361,48.360578
5,U1-61,0.436384,0.021137,secondary,Rosenaustraße,50,augsburg,1,67,10.889344,48.360556
6,U1-42,0.099691,0.039163,tertiary,Gögginger Straße,50,augsburg,1,0,10.890404,48.360719
7,U1-41,0.100387,0.040349,tertiary,Gögginger Straße,50,augsburg,1,1,10.890384,48.360742
8,U1-2,0.092166,0.038131,tertiary,Gögginger Straße,50,augsburg,1,69,10.890419,48.360603
9,U1-3,0.473093,0.015152,tertiary,Stettenstraße,50,augsburg,1,365,10.891241,48.360762


In [9]:
luzern_detectors = con.sql("""
    SELECT *
    FROM read_csv(
        ?,
        header = true,
        delim = ',',
        quote = '"',
        all_varchar = true
    )
    WHERE lower(citycode) = 'luzern'
""", params=[str(DATA_DIR / "detectors.csv")]).df()

print(f"Luzern detectors: {len(luzern_detectors)}")
display(luzern_detectors.head(10))

Luzern detectors: 158


,detid,length,pos,fclass,road,limit,citycode,lanes,linkid,long,lat
0,ig11FD116_D20,0.117622605755175,0.042476931131866,primary,Baselstrasse,0,luzern,1,91,8.2970662,47.0521407
1,ig11FD116_D15,0.084304613395923,0.043320630438901,primary,Baselstrasse,0,luzern,1,92,8.2970381,47.0521078
2,ig11FD116_D35,0.268815714824,0.22362392624425,primary,Baselstrasse,0,luzern,1,90,8.2971145,47.0521918
3,ig11FD116_D34,0.11348333835617,0.068332388524665,primary,Baselstrasse,0,luzern,1,89,8.2971293,47.0522229
4,ig11FD116_D30,0.097651051543157,0.034251934100282,primary,Gütschstrasse,50,luzern,1,83,8.2982062,47.0523883
5,ig11FD116_D25,0.098785204475424,0.034766390660782,primary,Gütschstrasse,50,luzern,1,82,8.298174,47.0524038
6,ig11FD116_D33,0.170677071363512,0.108890889782684,primary,Baselstrasse,50,luzern,1,54,8.298866,47.0518035
7,ig11FD100_D31,0.204483145661009,0.155278235457219,primary,Seebrücke,50,luzern,1,1,8.3092234,47.0514343
8,ig11FD100_D32,0.202738339518537,0.174769420257969,primary,Seebrücke,50,luzern,1,2,8.3091731,47.0514375
9,ig11FD100_D33,0.116316471448625,0.103917994520274,primary,Bahnhofplatz,50,luzern,1,6,8.3090309,47.0501258


In [10]:
# Convert coordinates from text to numeric values
luzern_detectors["lat"] = pd.to_numeric(
    luzern_detectors["lat"], errors="coerce"
)

luzern_detectors["long"] = pd.to_numeric(
    luzern_detectors["long"], errors="coerce"
)

print("Metadata records:", len(luzern_detectors))
print("Unique detectors:", luzern_detectors["detid"].nunique())

print(
    "Detectors with coordinates:",
    luzern_detectors[["lat", "long"]].notna().all(axis=1).sum()
)

print(
    "Detectors without coordinates:",
    luzern_detectors[["lat", "long"]].isna().any(axis=1).sum()
)

Metadata records: 158
Unique detectors: 158
Detectors with coordinates: 158
Detectors without coordinates: 0


In [13]:
import folium

# Center the map on Luzern
m = folium.Map(
    location=[
        luzern_detectors["lat"].mean(),
        luzern_detectors["long"].mean()
    ],
    zoom_start=13
)

# Add each detector to the map
for _, detector in luzern_detectors.iterrows():
    folium.CircleMarker(
        location=[detector["lat"], detector["long"]],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        popup=(
            f"Detector: {detector['detid']}<br>"
            f"Road: {detector['road']}<br>"
            f"Link: {detector['linkid']}"
        )
    ).add_to(m)

display(m)

In [14]:
links = con.sql("""
    SELECT *
    FROM read_csv(
        ?,
        header = true,
        delim = ',',
        quote = '"',
        all_varchar = true
    )
    WHERE lower(citycode) = 'luzern'
""", params=[str(DATA_DIR / "links.csv")]).df()

print("Road geometry records:", len(links))
print("Unique road links:", links["linkid"].nunique())

display(links.head(10))

Road geometry records: 612
Unique road links: 142


,long,lat,order,piece,linkid,group,citycode
0,8.3102426,47.0510871,1,1,0,0.1,luzern
1,8.3098362,47.0510524,2,1,0,0.1,luzern
2,8.3095989,47.051056,3,1,0,0.1,luzern
3,8.3094299,47.0511063,4,1,0,0.1,luzern
4,8.3094192,47.0510168,1,1,1,1.1,luzern
5,8.3093105,47.0511511,2,1,1,1.1,luzern
6,8.3092555,47.0513274,3,1,1,1.1,luzern
7,8.3092032,47.0516335,4,1,1,1.1,luzern
8,8.3090048,47.0528232,5,1,1,1.1,luzern
9,8.3093508,47.0510231,1,1,2,2.1,luzern


In [15]:
# Select one road link
road = links[links["linkid"] == "1"].copy()

# Convert coordinates and ordering
for col in ["lat", "long", "order"]:
    road[col] = pd.to_numeric(road[col])

# Sort points in traffic-flow direction
road = road.sort_values(["piece", "group", "order"])

# Extract coordinates for Folium
coordinates = road[["lat", "long"]].values.tolist()

# Create map
m = folium.Map(
    location=coordinates[0],
    zoom_start=16
)

folium.PolyLine(
    locations=coordinates,
    color="red",
    weight=5
).add_to(m)

folium.Marker(
    coordinates[0],
    popup="First geometry point",
    icon=folium.Icon(color="green")
).add_to(m)

folium.Marker(
    coordinates[-1],
    popup="Last geometry point",
    icon=folium.Icon(color="red")
).add_to(m)

display(m)

In [16]:
luzern_detectors[
    luzern_detectors["linkid"] == "1"
][["detid", "road", "linkid", "lat", "long"]]

,detid,road,linkid,lat,long
7,ig11FD100_D31,Seebrücke,1,47.051434,8.309223


In [17]:
luzern_detectors[
    luzern_detectors["road"] == "Seebrücke"
][["detid", "linkid", "lat", "long", "pos"]]

,detid,linkid,lat,long,pos
7,ig11FD100_D31,1,47.051434,8.309223,0.155278235457219
8,ig11FD100_D32,2,47.051437,8.309173,0.174769420257969
16,ig11FD100_D40,4,47.051158,8.309119,0.110052958139713
17,ig11FD100_D41,3,47.051250,8.309155,0.114076712228119
84,ig11FD201_D17,55,47.052747,8.308862,0.170976507888524
85,ig11FD201_D18,56,47.052752,8.308908,0.159402879128266


In [19]:
bridge = luzern_detectors[
    luzern_detectors["road"] == "Seebrücke"
].copy()

m = folium.Map(
    location=[47.0519, 8.3091],
    zoom_start=17
)

colors = ["red", "blue", "green", "purple", "orange", "black"]

for (_, detector), color in zip(bridge.iterrows(), colors):
    link = links[
        links["linkid"] == detector["linkid"]
    ].copy()

    for col in ["lat", "long", "order"]:
        link[col] = pd.to_numeric(link[col])

    link = link.sort_values(["piece", "group", "order"])

    coords = link[["lat", "long"]].values.tolist()

    if len(coords) < 2:
        continue

    folium.PolyLine(
        coords,
        color=color,
        weight=4,
        tooltip=f"Link {detector['linkid']}"
    ).add_to(m)

    folium.CircleMarker(
        location=[detector["lat"], detector["long"]],
        radius=5,
        color=color,
        fill=True,
        tooltip=detector["detid"]
    ).add_to(m)

    folium.Marker(
        coords[-1],
        tooltip=f"End of link {detector['linkid']}",
        icon=folium.Icon(color="red", icon="arrow-up")
    ).add_to(m)

display(m)

In [20]:
for _, detector in bridge.iterrows():
    link = links[
        links["linkid"] == detector["linkid"]
    ].copy()

    for col in ["lat", "long", "order"]:
        link[col] = pd.to_numeric(link[col])

    link = link.sort_values(["piece", "group", "order"])

    if len(link) < 2:
        continue

    start = link.iloc[0]
    end = link.iloc[-1]

    direction = (
        "Northbound" if end["lat"] > start["lat"]
        else "Southbound"
    )

    print(
        f"{detector['detid']} | "
        f"Link {detector['linkid']} | "
        f"{direction}"
    )

ig11FD100_D31 | Link 1 | Northbound
ig11FD100_D32 | Link 2 | Northbound
ig11FD100_D40 | Link 4 | Southbound
ig11FD100_D41 | Link 3 | Southbound
ig11FD201_D17 | Link 55 | Southbound
ig11FD201_D18 | Link 56 | Southbound


In [21]:
bridge[[
    "detid",
    "road",
    "linkid",
    "length",
    "pos",
    "lanes"
]].sort_values("linkid")

,detid,road,linkid,length,pos,lanes
7,ig11FD100_D31,Seebrücke,1,0.204483145661009,0.155278235457219,1
8,ig11FD100_D32,Seebrücke,2,0.202738339518537,0.174769420257969,1
17,ig11FD100_D41,Seebrücke,3,0.117499561589377,0.114076712228119,1
16,ig11FD100_D40,Seebrücke,4,0.11323368801833,0.110052958139713,1
84,ig11FD201_D17,Seebrücke,55,0.199870758669144,0.170976507888524,1
85,ig11FD201_D18,Seebrücke,56,0.187404162928914,0.159402879128266,1


In [22]:
from math import radians, sin, cos, asin, sqrt

def distance_m(lat1, lon1, lat2, lon2):
    """Calculate approximate distance between coordinates in meters."""
    R = 6371000

    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = (
        sin(dlat / 2)**2
        + cos(radians(lat1))
        * cos(radians(lat2))
        * sin(dlon / 2)**2
    )

    return 2 * R * asin(sqrt(a))


reference = bridge[
    bridge["detid"] == "ig11FD100_D31"
].iloc[0]

nearby = luzern_detectors.copy()

nearby["distance_m"] = nearby.apply(
    lambda row: distance_m(
        reference["lat"],
        reference["long"],
        row["lat"],
        row["long"]
    ),
    axis=1
)

nearby = nearby[
    (nearby["distance_m"] <= 300)
    & (nearby["detid"] != reference["detid"])
].sort_values("distance_m")

display(
    nearby[[
        "detid", "road", "linkid", "distance_m"
    ]]
)

,detid,road,linkid,distance_m
8,ig11FD100_D32,Seebrücke,2,3.827391
17,ig11FD100_D41,Seebrücke,3,21.159833
16,ig11FD100_D40,Seebrücke,4,31.687574
19,ig11FD100_D39,Bahnhofstrasse,5,49.691238
18,ig11FD100_D42,Bahnhofplatz,0,50.367163
9,ig11FD100_D33,Bahnhofplatz,6,146.227677
14,ig11FD100_D38,Bahnhofplatz,13,146.420876
85,ig11FD201_D18,Seebrücke,56,148.443769
84,ig11FD201_D17,Seebrücke,55,148.556433
15,ig11FD100_D37,Bahnhofplatz,14,150.640785


In [23]:
# Select our reference detector and nearby detectors
selected_ids = set(nearby["detid"])
selected_ids.add("ig11FD100_D31")

selected = luzern_detectors[
    luzern_detectors["detid"].isin(selected_ids)
].copy()

m = folium.Map(
    location=[47.0510, 8.3092],
    zoom_start=17
)

# Plot each road geometry
for linkid, group in links[
    links["linkid"].isin(selected["linkid"])
].groupby("linkid"):

    for _, piece in group.groupby(["piece", "group"]):
        piece = piece.copy()

        for col in ["lat", "long", "order"]:
            piece[col] = pd.to_numeric(piece[col])

        piece = piece.sort_values("order")

        if len(piece) < 2:
            continue

        folium.PolyLine(
            piece[["lat", "long"]].values.tolist(),
            color="blue",
            weight=3,
            opacity=0.6,
            tooltip=f"Link {linkid}"
        ).add_to(m)

# Plot sensor locations
for _, detector in selected.iterrows():
    folium.CircleMarker(
        location=[detector["lat"], detector["long"]],
        radius=5,
        color="red",
        fill=True,
        fill_opacity=0.9,
        popup=(
            f"Detector: {detector['detid']}<br>"
            f"Road: {detector['road']}<br>"
            f"Link: {detector['linkid']}"
        )
    ).add_to(m)

display(m)

In [24]:
display(
    selected[
        ["detid", "road", "linkid", "lat", "long"]
    ].sort_values(["road", "linkid"])
)

,detid,road,linkid,lat,long
18,ig11FD100_D42,Bahnhofplatz,0,47.051064,8.309607
14,ig11FD100_D38,Bahnhofplatz,13,47.050119,8.309148
15,ig11FD100_D37,Bahnhofplatz,14,47.050080,8.309167
9,ig11FD100_D33,Bahnhofplatz,6,47.050126,8.309031
19,ig11FD100_D39,Bahnhofstrasse,5,47.051041,8.308911
22,ig11FD103_D3,Pilatusstrasse,15,47.049518,8.307546
26,ig11FD104_D5,Pilatusstrasse,16,47.049255,8.306955
23,ig11FD103_D2,Pilatusstrasse,18,47.049423,8.307421
24,ig11FD103_D6,Pilatusstrasse,6,47.049815,8.308287
21,ig11FD100_D44,Robert-Zünd-Strasse,11,47.049951,8.311401


In [25]:
candidate_roads = [
    "Zentralstrasse",
    "Bahnhofplatz",
    "Seebrücke"
]

candidates = selected[
    selected["road"].isin(candidate_roads)
].copy()

for _, detector in candidates.iterrows():
    link = links[
        links["linkid"] == detector["linkid"]
    ].copy()

    link[["lat", "long", "order"]] = link[
        ["lat", "long", "order"]
    ].apply(pd.to_numeric)

    link = link.sort_values(["piece", "group", "order"])

    if len(link) < 2:
        continue

    start = link.iloc[0]
    end = link.iloc[-1]

    print(
        f"{detector['detid']} | "
        f"{detector['road']} | "
        f"Link {detector['linkid']} | "
        f"Start: ({start['lat']:.5f}, {start['long']:.5f}) | "
        f"End: ({end['lat']:.5f}, {end['long']:.5f})"
    )

ig11FD100_D31 | Seebrücke | Link 1 | Start: (47.05102, 8.30942) | End: (47.05282, 8.30900)
ig11FD100_D32 | Seebrücke | Link 2 | Start: (47.05102, 8.30935) | End: (47.05282, 8.30895)
ig11FD100_D33 | Bahnhofplatz | Link 6 | Start: (47.05018, 8.30917) | End: (47.04965, 8.30786)
ig11FD100_D3 | Zentralstrasse | Link 9 | Start: (47.04586, 8.30926) | End: (47.04995, 8.30939)
ig11FD100_D4 | Zentralstrasse | Link 10 | Start: (47.04994, 8.30933) | End: (47.04593, 8.30905)
ig11FD100_D34 | Zentralstrasse | Link 7 | Start: (47.04995, 8.30939) | End: (47.05019, 8.30938)
ig11FD100_D35 | Zentralstrasse | Link 8 | Start: (47.04994, 8.30933) | End: (47.05020, 8.30934)
ig11FD100_D38 | Bahnhofplatz | Link 13 | Start: (47.05012, 8.30915) | End: (47.05035, 8.30944)
ig11FD100_D37 | Bahnhofplatz | Link 14 | Start: (47.05007, 8.30912) | End: (47.05029, 8.30946)
ig11FD100_D40 | Seebrücke | Link 4 | Start: (47.05119, 8.30911) | End: (47.05019, 8.30919)
ig11FD100_D41 | Seebrücke | Link 3 | Start: (47.05128, 8.309

In [26]:
pair_coverage = con.sql("""
    SELECT
        detid,
        COUNT(*) AS observations,
        COUNT(DISTINCT day) AS observed_days,
        MIN(day) AS first_day,
        MAX(day) AS last_day
    FROM read_parquet(?)
    WHERE detid IN (
        'ig11FD100_D3',
        'ig11FD100_D34',
        'ig11FD100_D38'
    )
      AND error IS NULL
      AND flow IS NOT NULL
    GROUP BY detid
    ORDER BY detid
""", params=[
    str(DATA_DIR / "luzern_raw.parquet")
]).df()

display(pair_coverage)

,detid,observations,observed_days,first_day,last_day
0,ig11FD100_D3,143964,348,2015-01-01,2015-12-31
1,ig11FD100_D34,162685,361,2015-01-01,2015-12-31
2,ig11FD100_D38,155153,361,2015-01-01,2015-12-31


In [27]:
aligned = con.sql("""
    SELECT
        day,
        interval,
        MAX(CASE WHEN detid = 'ig11FD100_D3'
            THEN flow END) AS D3,
        MAX(CASE WHEN detid = 'ig11FD100_D34'
            THEN flow END) AS D34,
        MAX(CASE WHEN detid = 'ig11FD100_D38'
            THEN flow END) AS D38
    FROM read_parquet(?)
    WHERE detid IN (
        'ig11FD100_D3',
        'ig11FD100_D34',
        'ig11FD100_D38'
    )
      AND error IS NULL
      AND flow IS NOT NULL
      AND interval >= 0
      AND interval < 86400
    GROUP BY day, interval
    ORDER BY day, interval
""", params=[
    str(DATA_DIR / "luzern_raw.parquet")
]).df()

print("Total timestamps:", len(aligned))
print("Complete timestamps:", len(aligned.dropna()))

display(aligned.head())

Total timestamps: 169771
Complete timestamps: 131672


,day,interval,D3,D34,D38
0,2015-01-01,180,180,120,200
1,2015-01-01,360,100,100,240
2,2015-01-01,540,160,160,<NA>
3,2015-01-01,720,<NA>,<NA>,180
4,2015-01-01,900,<NA>,100,80


In [28]:
# Correlation between the three detectors
correlation_matrix = aligned[
    ["D3", "D34", "D38"]
].astype(float).corr(method="pearson")

display(correlation_matrix)

,D3,D34,D38
D3,1.000000,0.886762,0.68134
D34,0.886762,1.000000,0.68021
D38,0.681340,0.680210,1.00000


In [29]:
data = aligned.copy()

data["timestamp"] = (
    pd.to_datetime(data["day"])
    + pd.to_timedelta(data["interval"], unit="s")
)

data = data.set_index("timestamp").sort_index()

for minutes in [0, 3, 6, 9, 12]:
    future_d34 = data["D34"].shift(
        freq=f"-{minutes}min"
    )

    correlation = data["D3"].astype(float).corr(
        future_d34.astype(float)
    )

    print(f"Delay {minutes:2d} min: {correlation:.4f}")

Delay  0 min: 0.8868
Delay  3 min: 0.7724
Delay  6 min: 0.7019
Delay  9 min: 0.6887
Delay 12 min: 0.6851
